In [30]:
import nltk
from string import punctuation
from heapq import nlargest
import re
import sys

class SimpleTokenizer:
    """Fallback tokenizer when NLTK is not available."""
    
    @staticmethod
    def sentence_tokenize(text):
        """Split text into sentences using regex."""
        # Split on period followed by space and capital letter
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
        return [s.strip() for s in sentences if s.strip()]
    
    @staticmethod
    def word_tokenize(text):
        """Split text into words using regex."""
        # Remove punctuation and split on whitespace
        text = re.sub(r'[^\w\s]', ' ', text)
        return [word.strip().lower() for word in text.split() if word.strip()]

class TextSummarizer:
    def __init__(self):
        """Initialize the summarizer with either NLTK or simple tokenizer."""
        self.using_nltk = False
        try:
            # Try to initialize NLTK
            print("Attempting to initialize NLTK...")
            nltk.download('punkt', quiet=True)
            nltk.download('stopwords', quiet=True)
            self.stopwords = set(nltk.corpus.stopwords.words('english') + list(punctuation))
            # Test NLTK tokenization
            test = nltk.sent_tokenize("Test sentence. Another test.")
            self.using_nltk = True
            print("Successfully initialized NLTK.")
        except Exception as e:
            print(f"NLTK initialization failed: {str(e)}")
            print("Falling back to simple tokenizer.")
            self.stopwords = set(list(punctuation) + ['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'is', 'are', 'was', 'were', 'will', 'be'])
        
        self.tokenizer = nltk if self.using_nltk else SimpleTokenizer
    
    def preprocess_text(self, text):
        """Clean and tokenize the text."""
        try:
            # Ensure text is a string
            if not isinstance(text, str):
                text = str(text)
            
            # Tokenize into sentences
            sentences = (nltk.sent_tokenize(text) if self.using_nltk 
                       else self.tokenizer.sentence_tokenize(text))
            
            # Tokenize into words and remove stopwords
            words = (nltk.word_tokenize(text.lower()) if self.using_nltk 
                    else self.tokenizer.word_tokenize(text.lower()))
            words = [word for word in words if word not in self.stopwords]
            
            return sentences, words
            
        except Exception as e:
            print(f"Error in text preprocessing: {str(e)}")
            raise
    
    def calculate_word_frequencies(self, words):
        """Calculate word frequencies."""
        word_freq = {}
        for word in words:
            word_freq[word] = word_freq.get(word, 0) + 1
        
        # Normalize frequencies
        max_freq = max(word_freq.values()) if word_freq else 1
        for word in word_freq:
            word_freq[word] = word_freq[word] / max_freq
            
        return word_freq
    
    def score_sentences(self, sentences, word_freq):
        """Score sentences based on word frequencies."""
        sentence_scores = {}
        
        for sentence in sentences:
            words = (nltk.word_tokenize(sentence.lower()) if self.using_nltk 
                    else self.tokenizer.word_tokenize(sentence.lower()))
            
            # Skip very short sentences
            if len(words) <= 3:
                continue
            
            # Calculate score based on word frequencies
            score = sum(word_freq.get(word, 0) for word in words)
            # Normalize by sentence length
            sentence_scores[sentence] = score / len(words)
            
        return sentence_scores
    
    def summarize(self, text, num_sentences=3):
        """Generate a summary of the input text."""
        if not text or num_sentences <= 0:
            return ""
            
        try:
            # Preprocess the text
            sentences, words = self.preprocess_text(text)
            
            if not sentences:
                return ""
                
            # Ensure we don't try to return more sentences than exist
            num_sentences = min(num_sentences, len(sentences))
            
            # Calculate word frequencies
            word_freq = self.calculate_word_frequencies(words)
            
            # Score sentences
            sentence_scores = self.score_sentences(sentences, word_freq)
            
            # Select top sentences
            summary_sentences = nlargest(num_sentences, sentence_scores, key=sentence_scores.get)
            
            # Reorder sentences based on their original position
            summary_sentences.sort(key=sentences.index)
            
            # Join sentences to create the summary
            summary = ' '.join(summary_sentences)
            
            return summary
            
        except Exception as e:
            print(f"Error generating summary: {str(e)}")
            raise

def main():
    # Example usage
    example_text = """
    Artificial intelligence has transformed numerous industries in recent years. 
    Machine learning algorithms are now capable of processing vast amounts of data and making complex decisions. 
    Natural language processing has enabled computers to understand and generate human-like text. 
    Computer vision systems can now recognize objects and faces with remarkable accuracy. 
    These advances have led to applications in healthcare, finance, and autonomous vehicles. 
    However, concerns about AI safety and ethics continue to be important topics of discussion. 
    Researchers are working to ensure AI systems remain transparent and accountable. 
    The future of AI holds both exciting possibilities and important challenges to address.
    """
    
    try:
        print("Initializing summarizer...")
        summarizer = TextSummarizer()
        
        print("\nGenerating summary...")
        summary = summarizer.summarize(example_text, num_sentences=3)
        
        print("\nOriginal Text:")
        print(example_text.strip())
        print("\nSummary:")
        print(summary)
        
    except Exception as e:
        print(f"\nError: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Initializing summarizer...
Attempting to initialize NLTK...
Successfully initialized NLTK.

Generating summary...

Original Text:
Artificial intelligence has transformed numerous industries in recent years. 
    Machine learning algorithms are now capable of processing vast amounts of data and making complex decisions. 
    Natural language processing has enabled computers to understand and generate human-like text. 
    Computer vision systems can now recognize objects and faces with remarkable accuracy. 
    These advances have led to applications in healthcare, finance, and autonomous vehicles. 
    However, concerns about AI safety and ethics continue to be important topics of discussion. 
    Researchers are working to ensure AI systems remain transparent and accountable. 
    The future of AI holds both exciting possibilities and important challenges to address.

Summary:
Natural language processing has enabled computers to understand and generate human-like text. Researchers are